In [ ]:
# 01 · STATE STAGE ACT 단일 정책 CONFIG · GitHub 중단 복구 · Drive 미사용
CFG = {
    # 저장 · ver2는 기존 DP 결과와 별도 Release에 저장
    'run_name': 'moveboxes_stage_pick_residual_v1',
    'profile': 'benchmark',
    'github_repository': 'SongYunu/moveBoxes',
    'project_dir': '/content/moveBoxes',
    'project_ref': 'b67d8ba6ddcf1d514a33ca2e82859a534720dc47',
    'output_root': '/content/moveboxes_runs',

    # 데이터 / Colab 2026.07 · Python 3.12 · T4
    'repo_dir': '/content/berlin-marso-hackathon',
    'repo_url': 'https://github.com/marso-robotics/berlin-marso-hackathon.git',
    'repo_commit': '6048f33217f26ae39009a812f53c81171517f393',
    'data_dir': '/content/marso_data',
    'data_source': '/content/moveboxes_data_cache/marso_state_data.zip',
    'download_cache': '/content/moveboxes_data_cache',
    'packages': ['mani-skill==3.0.1', 'sapien==3.0.3', 'diffusers==0.38.0', 'hydra-core', 'omegaconf', 'gymnasium', 'tyro', 'h5py', 'kagglehub', 'tensorboard', 'matplotlib', 'transforms3d', 'imageio[ffmpeg]'],

    # 학습 · 시뮬레이터 없이 CPU 데이터 + GPU 모델
    'seed': 42,
    'num_demos': None,
    'batch_size': 64,
    'lr': 0.0001,
    'total_iters': {'easy': 12000, 'medium': 20000, 'hard': 30000},
    'amp': True,

    # 작은 State ACT · 기존 DP 체크포인트 사용 불가
    'history': 16,
    'chunk_size': 16,
    'width': 128,
    'heads': 4,
    'layers': 2,
    'latent_dim': 16,

    # 검증 / 중단 복구 / 작은 관측 위치 증강
    'save_freq': 1000,
    'warmup_steps': 500,
    'validation_batches': 8,
    'kl_weight': 0.001,
    'position_noise': 0.001,

    # 실행 · 매 스텝 재계획, 최근 XYZ 예측 평균, 집게는 최신 예측
    'temporal_decay': 0.25,
    'ensemble_window': 4,
    'ensemble_candidates': [1, 4],

    # 빠른 테스트 / 최종 평가 · 시드 분리, 기존 200스텝 유지
    'test_episodes': 8,
    'test_seed_start': 40000,
    'test_record_video': True,
    'tuning_episodes': 8,
    'tuning_seed_start': 20000,
    'benchmark_episodes': 100,
    'eval_seed_start': 30000,
    'max_episode_steps': {'easy': 200, 'medium': 200, 'hard': 200},
    'record_eval_video': True,

    # 출력
    'console_interval_seconds': 10,
    'team': 'my-team',

    # 실행 조건으로 행동 학습 · 빠른 테스트가 0이면 긴 평가 생략
    'action_training_mode': 'prior',
    'repair_iters': 2000,
    'allow_zero_success_evaluation': False,

    # 단계 판단 · 학습된 완료/복구 확신이 낮으면 현재 단계 유지
    'gate_threshold': 0.65,
    'stage_threshold': 0.6,
    'stage_loss_weight': 0.3,
    'gate_loss_weight': 0.3,

    # 복구 시연 · 수집 전용 expert, 학습/제출은 학습된 정책
    'recovery_episodes': 16,
    'recovery_max_attempts': 48,
    'recovery_seed_start': 100000,
    'collection_max_steps': {'easy': 500, 'medium': 900, 'hard': 1400},
    'noise_probability': 0.08,
    'action_noise_std': 0.12,
    'drop_probability': 0.015,

}


In [ ]:
# 02 · GitHub 코드 불러오기 (데이터·결과를 위해 Drive를 마운트하지 않습니다)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
URL = 'https://github.com/'+CFG['github_repository']+'.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if remote != URL:
        raise RuntimeError('기존 프로젝트 폴더가 다른 저장소입니다. project_dir를 새 경로로 바꾸세요.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CFG['project_ref']], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
CFG['project_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
sys.path.insert(0, str(PROJECT))
# A fresh notebook run should not retain a previously imported project module.
for name in ('marso_experiment', 'marso_train_test', 'next_pick_sampling', 'next_pick_diagnostics',
             'marso_next_pick', 'github_store', 'github_data', 'colab_layout', 'build_modular_notebook',
             'colab_train_test_layout', 'build_train_test_notebook', 'colab_next_pick_layout',
             'build_next_pick_notebook', 'build_github_notebook', 'marso_github'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'))
for name in ('act_v2_model','act_v2_data','act_v2_policy','act_v2_eval','act_v2_experiment','build_act_v2_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'/'stages'))
for name in ('stage_schema','stage_model','stage_policy','stage_labels','stage_data','stage_teacher',
             'stage_collect','stage_eval','stage_experiment','stage_chunk_policy',
             'stage_pick_sampling','stage_pick_train','stage_all_pick_retrain',
             'stage_pick_finetune','stage_pick_diagnose','stage_reference_check',
             'stage_anchor_continue','build_stage_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
from stage_experiment import StageExperiment, source_bundle
experiment = StageExperiment(CFG, source_bundle())
print('사용 코드:', CFG['project_commit'])
print('코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.')


In [ ]:
# 03 · GitHub 인증 / 저장된 결과 복원
# 기존 환경변수 GH_TOKEN → Colab 보안 비밀 → 입력창 순서로 인증합니다.
# Fine-grained token: SongYunu/moveBoxes → Contents: Read and write.
# 이 저장소는 공개이므로 여기에 올린 모델·로그·영상도 공개됩니다.
import os
# 개인 사본에서 직접 지정할 경우 아래 한 줄의 주석을 풀어 사용하세요.
# os.environ['GH_TOKEN'] = '본인 토큰'
experiment.connect()
experiment.show_results()


In [ ]:
# 04 · 새 런타임마다 환경 설치
experiment.install()


In [ ]:
# 05 · GitHub 데이터 다운로드·검증 / GPU와 정책 실행 확인
experiment.prepare_data()
experiment.check_runtime()


# Stage ACT 앵커 + PICK Residual Group RL

검증된 난이도별 Stage ACT 앵커를 그대로 복원하고 **전체 앵커를 동결**합니다. 새 정책은 `PICK` 단계의 XYZ에만 작은 상태 기반 residual을 더합니다. residual의 마지막 층은 0으로 초기화되므로 학습 전 동작은 앵커와 정확히 같습니다.

각 초기 배치 seed마다 4개의 stochastic 집기 후보를 실행하고, 같은 배치 안에서 실제 `success_count`가 더 높은 후보에만 상대 advantage를 줍니다. critic과 demonstration target은 사용하지 않습니다. 한 집기 시도의 마지막 12 step만 업데이트하므로 운반·놓기와 긴 대기 동작이 집기 보정 gradient를 오염시키지 않습니다.

8회마다 공식 `eval.py`를 실행합니다. 최고 공식 점수는 별도 파일로 보존하고, 3번 연속 개선이 없으면 자동으로 멈춥니다. Drive는 사용하지 않으며 iteration checkpoint와 optimizer는 GitHub Release에 복구됩니다.


In [ ]:
# 06 · 검증 앵커 복원 / 공식 평가 함수
import json, os, re, shutil, subprocess, sys
from pathlib import Path
from IPython.display import Video, display
from stage_anchor_continue import (ANCHORS, package, prepare, prepare_pick_residual_rl,
                                   train_pick_residual_rl)

MAX_STEPS = 200
MEDIUM_GROUP_ITERATIONS = 64
RL_EVAL_EVERY = 8
EARLY_STOP_EVALS = 3
UPSTREAM = Path(CFG['repo_dir'])
OFFICIAL_DEFAULT = UPSTREAM/'conf/eval/default.yaml'
SELECTION = Path(CFG['output_root'])/'pick_residual_selection_eval.yaml'
SELECTION.write_text('eval:\n  n_episodes: 8\n  seeds: '+json.dumps(list(range(63000, 63008)))+'\n', encoding='utf-8')
anchor_exp = prepare(experiment, run_suffix='_anchor_pick_residual_base_v1')
RUN_DIR = Path(anchor_exp.run_dir)
BASELINE = package(anchor_exp, folder_name='anchor_candidate')

def run_official(candidate, level, label, eval_config=SELECTION):
    output = RUN_DIR/level/'pick_residual_official_eval'/candidate.name/label
    output.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(UPSTREAM/'eval.py'), 'difficulty='+level,
        'obs_mode=state', 'policy=stage_policy:load_policy',
        'checkpoint='+str(candidate/'checkpoints'/level/'model.pt'),
        'eval_config='+str(eval_config), 'max_episode_steps='+str(MAX_STEPS),
        'hydra.run.dir='+str(output)]
    child_env = dict(os.environ)
    child_env['PYTHONPATH'] = str(candidate)+os.pathsep+str(UPSTREAM)+os.pathsep+child_env.get('PYTHONPATH','')
    log = output/'official_eval.log'
    with log.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(command, cwd=UPSTREAM, env=child_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            errors='replace', bufsize=1)
        try:
            for line in process.stdout:
                handle.write(line); handle.flush(); print(line, end='', flush=True)
            code = process.wait()
        finally:
            if process.poll() is None:
                process.terminate()
                try: process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill(); process.wait()
            process.stdout.close()
    if code:
        raise RuntimeError(f'official eval failed ({level}); {log} 확인')
    match = re.search(r'SORT ACCURACY:\s+([0-9.]+)\s*%', log.read_text(encoding='utf-8'))
    if not match:
        raise ValueError(f'공식 점수를 읽지 못했습니다: {log}')
    videos = sorted((output/'videos').rglob('*.mp4'), key=lambda p:p.stat().st_mtime)
    if videos:
        display(Video(str(videos[-1]), embed=True, width=900))
    return {'level':level, 'score':float(match.group(1))/100, 'log':str(log),
            'checkpoint':str(candidate/'checkpoints'/level/'model.pt')}


In [ ]:
# 07 · Medium 학습 전 8-seed 선택 평가 + 영상
BASELINE_RESULTS = {'medium':run_official(BASELINE, 'medium', 'anchor_selection')}
print(json.dumps(BASELINE_RESULTS, indent=2))


## Medium PICK residual 학습

`GROUP RL` 로그의 `sorted`는 해당 iteration의 16개 rollout이 분류한 상자 수이고, 모델 선택 점수는 매 8회 뒤 출력되는 공식 `SORT ACCURACY`입니다. 모든 동일-seed 후보 점수가 같으면 update를 건너뜁니다. 중단되어도 이 셀을 다시 실행하면 마지막 완전한 iteration에서 재개합니다.


In [ ]:
# 09 · Medium PICK residual · 8회마다 공식 평가 · 3회 미개선 자동 중단
def evaluate_group_round(rl_exp, level, stop, best):
    boundary = rl_exp.run_dir/level/'checkpoints'/f'iteration_{stop:04d}.pt'
    if not boundary.is_file():
        raise FileNotFoundError(f'라운드 체크포인트가 없습니다: {boundary}')
    candidate = package(rl_exp, checkpoint_overrides={level:boundary},
                        folder_name=f'{level}_pick_residual_round_{stop:02d}')
    result = run_official(candidate, level, f'pick_residual_round_{stop:02d}')
    result.update(iteration=stop, source='pick_residual_group_rl')
    history_path = rl_exp.run_dir/level/'official_rounds.json'
    history = json.loads(history_path.read_text()) if history_path.is_file() else []
    history = [item for item in history if item.get('iteration') != stop]
    history.append({k:v for k,v in result.items() if k != 'checkpoint'})
    history_path.write_text(json.dumps(history, indent=2), encoding='utf-8')
    improved = result['score'] > best['score']
    if improved:
        promoted = rl_exp.run_dir/level/'checkpoints'/'official_best.pt'
        shutil.copy2(result['checkpoint'], promoted)
        result['checkpoint'] = str(promoted)
        best = result
        print(f'{level} {stop}회: 새 최고 공식 점수 {result["score"]:.3%}')
    else:
        print(f'{level} {stop}회: {result["score"]:.3%}; 현재 최고 {best["score"]:.3%} 유지')
    try:
        rl_exp.sync_level(level)
    except Exception as error:
        print('GitHub 백업 지연; 로컬 최고 체크포인트는 유지합니다:', error)
    return best, improved

def evaluation_stops(total, every=RL_EVAL_EVERY):
    stops = list(range(every, total+1, every))
    if not stops or stops[-1] != total:
        stops.append(total)
    return stops

medium_group = prepare_pick_residual_rl(
    anchor_exp, 'medium', iterations=MEDIUM_GROUP_ITERATIONS,
    num_envs=16, group_size=4, lr=2e-5, xyz_std=.04, pick_credit_steps=12)
MEDIUM_BEST = dict(BASELINE_RESULTS['medium'], iteration=0, source='anchor')
stale_evals = 0
for stop in evaluation_stops(MEDIUM_GROUP_ITERATIONS):
    train_pick_residual_rl(medium_group, 'medium', until_iteration=stop)
    MEDIUM_BEST, improved = evaluate_group_round(medium_group, 'medium', stop, MEDIUM_BEST)
    stale_evals = 0 if improved else stale_evals+1
    if stale_evals >= EARLY_STOP_EVALS:
        print(f'공식 점수 {EARLY_STOP_EVALS}회 연속 미개선: 최고 체크포인트를 유지하고 중단합니다.')
        break
MEDIUM_USE_GROUP = MEDIUM_BEST['source'] == 'pick_residual_group_rl'
print('Medium 최종 선택:', MEDIUM_BEST)


In [ ]:
# 10 · 최고 공식 점수만 패키징 · 전체 난이도 재검증 · ZIP
selected = {'medium':Path(MEDIUM_BEST['checkpoint'])} if MEDIUM_USE_GROUP else {}
FINAL = package(anchor_exp, checkpoint_overrides=selected,
                folder_name='final_pick_residual_candidate')
FINAL_RESULTS = {level:run_official(FINAL, level, 'final_default', OFFICIAL_DEFAULT)
                 for level in ('easy','medium','hard')}
print(json.dumps({'selected':{k:str(v) for k,v in selected.items()},
                  'official_results':FINAL_RESULTS}, indent=2))
archive = shutil.make_archive(str(RUN_DIR/'stage_act_pick_residual_submission'),
                              'zip', root_dir=FINAL)
if MEDIUM_USE_GROUP:
    demo_dir = Path(MEDIUM_BEST['log']).parent/'videos'
    demo_videos = sorted(demo_dir.rglob('*.mp4'), key=lambda p:p.stat().st_mtime)
    if demo_videos:
        print(f"Medium BEST-CASE DEMO · 공식 선택 회차 {MEDIUM_BEST['iteration']} · "
              f"해당 회차 SORT ACCURACY {MEDIUM_BEST['score']:.1%}")
        display(Video(str(demo_videos[-1]), embed=True, width=900))
from google.colab import files
files.download(archive)
